# Notebook 05 — Deep Learning Optimizers & Training Loop Hyperparameters

**Session:** Hyper-parameter optimization  
**Module:** 02 · Calculus and Linear Algebra

## Goal

Structure a clean, reproducible PyTorch CNN training and evaluation pipeline, inspect
key optimizer hyperparameters (learning rate $\eta$, Adam momentum coefficients
$\beta_1, \beta_2$, numerical epsilon $\epsilon$, and weight decay $\lambda$), and
implement the Adam update rule from scratch alongside `torch.optim.Adam`.

In [ ]:
import math

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

## 1. Synthetic Image Classification Dataset & DataLoader

In [ ]:
def make_synthetic_vision_dataset(
    n_samples: int = 128,
    image_size: int = 32,
    seed: int = 42,
) -> TensorDataset:
    """Create a portable 2-class 3-channel synthetic image dataset."""
    gen = torch.Generator().manual_seed(seed)
    images = torch.randn(n_samples, 3, image_size, image_size, generator=gen) * 0.3
    labels = torch.randint(0, 2, (n_samples,), generator=gen)
    # Inject class-dependent spatial signal so a CNN can learn above chance
    for idx in range(n_samples):
        if labels[idx] == 1:
            images[idx, 0, : image_size // 2, : image_size // 2] += 1.0
        else:
            images[idx, 2, image_size // 2 :, image_size // 2 :] += 1.0
    return TensorDataset(images, labels)


train_ds = make_synthetic_vision_dataset(n_samples=128, seed=42)
val_ds = make_synthetic_vision_dataset(n_samples=64, seed=99)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

## 2. Compact VGG-Style Convolutional Classifier

In [ ]:
class CompactVGG(nn.Module):
    """Compact VGG-style CNN for 32x32 RGB classification."""

    def __init__(self, n_classes: int = 2) -> None:
        super().__init__()
        self.features = nn.Sequential(
            self._conv_block(3, 16),
            self._conv_block(16, 32),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 8 * 8, 64),
            nn.ReLU(),
            nn.Linear(64, n_classes),
        )

    @staticmethod
    def _conv_block(in_channels: int, out_channels: int) -> nn.Sequential:
        return nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.classifier(self.features(x))

## 3. Custom Adam Optimizer Reference Implementation

In [ ]:
class CustomAdam(torch.optim.Optimizer):
    """Minimal from-scratch Adam optimizer with decoupled bias correction."""

    def __init__(
        self,
        params,
        lr: float = 1e-3,
        betas: tuple[float, float] = (0.9, 0.999),
        eps: float = 1e-8,
        weight_decay: float = 0.0,
    ) -> None:
        defaults = {"lr": lr, "betas": betas, "eps": eps, "weight_decay": weight_decay}
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure=None):
        loss = closure() if closure is not None else None
        for group in self.param_groups:
            beta1, beta2 = group["betas"]
            for p in group["params"]:
                if p.grad is None:
                    continue
                grad = p.grad
                if group["weight_decay"] != 0.0:
                    grad = grad.add(p, alpha=group["weight_decay"])

                state = self.state[p]
                if len(state) == 0:
                    state["step"] = 0
                    state["exp_avg"] = torch.zeros_like(p)
                    state["exp_avg_sq"] = torch.zeros_like(p)

                state["step"] += 1
                m, v = state["exp_avg"], state["exp_avg_sq"]
                m.mul_(beta1).add_(grad, alpha=1.0 - beta1)
                v.mul_(beta2).addcmul_(grad, grad, value=1.0 - beta2)

                bias_correction1 = 1.0 - beta1 ** state["step"]
                bias_correction2 = 1.0 - beta2 ** state["step"]
                step_size = group["lr"] / bias_correction1
                denom = (v.sqrt() / math.sqrt(bias_correction2)).add_(group["eps"])
                p.addcdiv_(m, denom, value=-step_size)
        return loss

## 4. Training & Validation Loop

In [ ]:
def evaluate(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
) -> tuple[float, float]:
    """Return (mean_loss, accuracy_percent) over *loader*."""
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            total_loss += criterion(logits, labels).item()
            correct += int((logits.argmax(dim=1) == labels).sum().item())
            total += labels.size(0)
    return total_loss / len(loader), 100.0 * correct / total


def train_model(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
    epochs: int,
    device: torch.device,
) -> dict[str, list[float]]:
    """Run full supervised training loop and record per-epoch metrics."""
    history: dict[str, list[float]] = {
        "train_loss": [],
        "val_loss": [],
        "train_acc": [],
        "val_acc": [],
    }
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(images), labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        train_loss, train_acc = evaluate(model, train_loader, criterion, device)
        val_loss, val_acc = evaluate(model, val_loader, criterion, device)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)
        print(
            f"Epoch {epoch + 1:2d}/{epochs} | "
            f"train_loss={train_loss:.4f} val_loss={val_loss:.4f} | "
            f"train_acc={train_acc:.1f}% val_acc={val_acc:.1f}%"
        )
    return history


torch.manual_seed(0)
model = CompactVGG(n_classes=2).to(device)
optimizer = CustomAdam(model.parameters(), lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.0)
criterion = nn.CrossEntropyLoss()

history = train_model(
    model, train_loader, val_loader, optimizer, criterion, epochs=5, device=device
)

plt.figure(figsize=(7, 4))
plt.plot(history["train_acc"], label="Train Accuracy")
plt.plot(history["val_acc"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.title("CompactVGG Training with CustomAdam")
plt.legend()
plt.tight_layout()
plt.show()